In [1]:
from pathlib import Path
from typing import Dict, List, Union
import sys
import json

import torch
from transformers import AutoModel, AutoTokenizer

# Add DMRST Parser to the Python import path
PROJECT_ROOT = Path.cwd().resolve().parent
DMRST_PATH = PROJECT_ROOT / 'external' / 'DMRST_Parser'
sys.path.insert(0, str(DMRST_PATH))

from model_depth import ParsingNet
from MUL_main_Infer import inference

d:\Dev\tcc\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class DMRSTParser:
    def __init__(
        self,
        checkpoint_path: Union[str, Path],
        cache_dir: Union[str, Path],
        batch_size: int = 1,
    ):
        if not torch.cuda.is_available():
            raise RuntimeError(
                'This DMRST_Parser version requires GPU with CUDA support.'
            )

        self.batch_size = batch_size
        self.checkpoint_path = Path(checkpoint_path)
        self.cache_dir = Path(cache_dir)

        self.cache_dir.mkdir(parents=True, exist_ok=True)

        if not self.checkpoint_path.exists():
            raise FileNotFoundError(
                f'Checkpoint not find: '
                f'{self.checkpoint_path.resolve()}'
            )

        self.tokenizer = AutoTokenizer.from_pretrained(
            'xlm-roberta-base',
            use_fast=True,
            cache_dir=str(self.cache_dir),
        )

        language_model = AutoModel.from_pretrained(
            'xlm-roberta-base',
            cache_dir=str(self.cache_dir),
        ).cuda()

        for parameter in language_model.parameters():
            parameter.requires_grad = False

        self.model = ParsingNet(
            language_model,
            bert_tokenizer=self.tokenizer,
        ).cuda()

        state_dict = torch.load(
            str(self.checkpoint_path),
            map_location="cuda",
        )

        if isinstance(state_dict, dict) and 'state_dict' in state_dict:
            state_dict = state_dict['state_dict']

        obsolete_keys = [
            key
            for key in state_dict
            if key.endswith('.embeddings.position_ids')
        ]

        for key in obsolete_keys:
            print(f'Ignoring old checkpoint key: {key}')
            state_dict.pop(key)

        self.model.load_state_dict(state_dict, strict=True)
        self.model.eval()

    def parse(
        self,
        texts: Union[str, List[str]],
    ) -> Union[Dict, List[Dict]]:
        received_single_text = isinstance(texts, str)

        if received_single_text:
            documents = [texts]
        else:
            documents = list(texts)

        documents = [document.strip() for document in documents]

        if not documents or any(not document for document in documents):
            raise ValueError('Text can\'t be empty')

        tokens_batch, breaks_batch, trees_batch = inference(
            model=self.model,
            tokenizer=self.tokenizer,
            input_sentences=documents,
            batch_size=self.batch_size,
        )

        results = []

        for original_text, tokens, edu_breaks, tree in zip(
            documents,
            tokens_batch,
            breaks_batch,
            trees_batch,
        ):
            edus = self._reconstruct_edus(tokens, edu_breaks)

            tree_value = (
                tree[0]
                if isinstance(tree, list) and len(tree) == 1
                else tree
            )

            results.append(
                {
                    'text': original_text,
                    'tokens': tokens,
                    'edu_breaks': edu_breaks,
                    'edus': edus,
                    'tree': tree_value,
                }
            )

        return results[0] if received_single_text else results

    def _reconstruct_edus(
        self,
        tokens: List[str],
        edu_breaks: List[int],
    ) -> List[str]:
        edus = []
        start = 0

        for end in edu_breaks:
            edu_tokens = tokens[start : end + 1]

            edu_text = self.tokenizer.convert_tokens_to_string(
                edu_tokens
            ).strip()

            edus.append(edu_text)
            start = end + 1

        return edus

In [3]:
parser = DMRSTParser(
    checkpoint_path='../artifacts/dmrst/multi_all_checkpoint.torchsave',
    cache_dir='../artifacts/models'
)

d:\Dev\tcc\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\Dev\tcc\artifacts\models\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2738.38it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key        

Ignoring old checkpoint key: encoder.language_model.embeddings.position_ids


In [6]:
rst = parser.parse('''
Google released Gemini 2.5 to improve reasoning and multimodal capabilities across its AI products.
Several months later, Google integrated Gemini 2.5 into Google Workspace, allowing users to generate documents, summarize emails, and analyze spreadsheets.
'''.strip())

In [7]:
rst

{'text': 'Google released Gemini 2.5 to improve reasoning and multimodal capabilities across its AI products.\nSeveral months later, Google integrated Gemini 2.5 into Google Workspace, allowing users to generate documents, summarize emails, and analyze spreadsheets.',
 'tokens': ['▁Google',
  '▁released',
  '▁Ge',
  'mini',
  '▁2.5',
  '▁to',
  '▁improve',
  '▁reason',
  'ing',
  '▁and',
  '▁multi',
  'mo',
  'dal',
  '▁capabil',
  'ities',
  '▁across',
  '▁its',
  '▁AI',
  '▁products',
  '.',
  '▁Sever',
  'al',
  '▁months',
  '▁later',
  ',',
  '▁Google',
  '▁integrat',
  'ed',
  '▁Ge',
  'mini',
  '▁2.5',
  '▁into',
  '▁Google',
  '▁Work',
  'space',
  ',',
  '▁',
  'allowing',
  '▁users',
  '▁to',
  '▁generate',
  '▁documents',
  ',',
  '▁summa',
  'riz',
  'e',
  '▁email',
  's',
  ',',
  '▁and',
  '▁anal',
  'y',
  'ze',
  '▁spread',
  'she',
  'ets',
  '.'],
 'edu_breaks': [4, 19, 35, 56],
 'edus': ['Google released Gemini 2.5',
  'to improve reasoning and multimodal capabilitie